In [4]:
# -*- coding: utf-8 -*-
"""
Monte Carlo giá điều WW320 – BẢN BUYER (2 ĐUÔI)
================================================
Cập nhật so với bản trước:
  1. Grid mu mở rộng sang DƯƠNG: [-8%, -4%, 0%, +4%, +8%]
     → Bao phủ cả kịch bản giá TĂNG (rủi ro cho buyer).
  2. Statistics tính CẢ 2 ĐUÔI:
     - Lower VaR/CVaR: đuôi dưới (cơ hội mua rẻ cho buyer)
     - Upper VaR/CVaR: đuôi trên (rủi ro chi phí cho buyer)
  3. Console in rõ 2 nhóm: RỦI RO (giá tăng) vs CƠ HỘI (giá giảm).
  4. Fan chart có 2 ngưỡng cho buyer:
     - Ngưỡng rủi ro chi phí (giá CAO, màu đỏ)
     - Ngưỡng cơ hội mua (giá THẤP, màu xanh)
  5. Tính xác suất P(giá > ngưỡng cao) và P(giá < ngưỡng thấp).
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# =================================================================
# 1. ĐƯỜNG DẪN  –  BẠN TỰ CHỈNH
# =================================================================
INPUT_PATH  = r"C:\Users\Acer\Desktop\Dan on food\Code\monte carlo\input.xlsx"
OUTPUT_PATH = r"C:\Users\Acer\Desktop\Dan on food\Code\monte carlo\ouput_buyer.xlsx"

RETURN_COLUMN = "% change "
PRICE_COLUMN  = "Average value"
INPUT_SHEET   = None

# =================================================================
# 2. THAM SỐ MONTE CARLO
# =================================================================
N_SIMS    = 50000
N_PERIODS = 12
DT        = 1 / 12
SEED      = 42

# =================================================================
# 3. LƯỚI QUÉT  –  MỞ RỘNG SANG MIỀN DƯƠNG
# =================================================================
# mu (drift năm): quét 5 mức bao phủ cả tăng và giảm
# → bắt được kịch bản giá TĂNG (rủi ro cho buyer)
MU_ANNUAL_GRID = [-0.08, -0.04, 0.00, +0.04, +0.08]

# sigma (vol năm): 5 mức xung quanh sigma lịch sử
SIG_FACTORS = [0.75, 0.875, 1.0, 1.25, 1.5]

# Kịch bản trung tâm cho fan chart: mu=0 (index 2), sig=1.0x (index 2)
CENTER_MU_IDX  = 2   # mu_annual = 0.00
CENTER_SIG_IDX = 2   # sig_factor = 1.0

# =================================================================
# 4. NGƯỠNG NGHIỆP VỤ CHO BUYER
# =================================================================
# Ngưỡng rủi ro chi phí: giá vượt mức này → biên gộp bị nén
RISK_HIGH_MULT = 1.18     # +18% so với S0

# Ngưỡng cơ hội mua: giá dưới mức này → cơ hội mua rẻ
OPPORTUNITY_LOW_MULT = 0.92   # -8% so với S0

# =================================================================
# 5. ĐỌC DỮ LIỆU
# =================================================================
def load_data(path, sheet, ret_col, price_col):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(path, sheet_name=sheet if sheet else 0)
    elif ext == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Định dạng không hỗ trợ: {ext}")

    if ret_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột returns: '{ret_col}'. Cột hiện có: {list(df.columns)}")
    if price_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột giá: '{price_col}'. Cột hiện có: {list(df.columns)}")

    returns = df[ret_col].dropna().astype(float).values
    prices  = df[price_col].dropna().astype(float).values

    if len(returns) < 12:
        raise ValueError("Cần ít nhất 12 điểm dữ liệu lịch sử.")
    return returns, prices

# =================================================================
# 6. KIỂM TRA MEAN REVERSION (AR1)
# =================================================================
def check_mean_reversion(returns):
    if len(returns) < 4:
        return
    rho = np.corrcoef(returns[:-1], returns[1:])[0, 1]
    print(f"[INFO] AR(1) autocorr returns: {rho:.3f}")
    if abs(rho) > 0.15:
        print(
            f"[WARN] |rho|={abs(rho):.3f} > 0.15 → chuỗi có xu hướng mean-reversion.\n"
            f"       GBM giả định bước độc lập. Kết quả vẫn dùng được nhưng\n"
            f"       mô hình Ornstein–Uhlenbeck sẽ phản ánh sát hơn đặc trưng giá điều."
        )

# =================================================================
# 7. GBM MỘT KỊCH BẢN
# =================================================================
def run_gbm(n_sims, n_periods, s0, mu_annual, sigma_annual, dt, rng):
    drift = (mu_annual - 0.5 * sigma_annual ** 2) * dt
    diff  = sigma_annual * np.sqrt(dt)

    z        = rng.standard_normal((n_periods, n_sims))
    log_inc  = drift + diff * z
    log_path = np.vstack([np.zeros(n_sims), np.cumsum(log_inc, axis=0)])
    paths    = s0 * np.exp(log_path)
    term_ret = paths[-1] / s0 - 1.0
    return paths, term_ret

# =================================================================
# 8. THỐNG KÊ 2 ĐUÔI
# =================================================================
def compute_stats(rets, conf_levels=(0.95, 0.99)):
    """
    Tính cả 2 đuôi:
      - Lower VaR/CVaR: đuôi dưới (return âm) → cơ hội mua rẻ cho buyer
      - Upper VaR/CVaR: đuôi trên (return dương) → rủi ro chi phí cho buyer
    """
    sorted_r = np.sort(rets)
    n = len(sorted_r)
    out = {
        "E[Return]"  : float(np.mean(rets)),
        "Std[Return]": float(np.std(rets, ddof=1)),
        "Median"     : float(np.median(rets)),
        "Min"        : float(sorted_r[0]),
        "Max"        : float(sorted_r[-1]),
    }
    for cl in conf_levels:
        alpha = 1 - cl
        pct   = int(round(cl * 100))

        # ─── ĐUÔI DƯỚI (cơ hội mua rẻ cho buyer) ───
        idx_low = max(1, int(np.floor(n * alpha)))
        out[f"Lower VaR {pct}%"]  = float(sorted_r[idx_low - 1])
        out[f"Lower CVaR {pct}%"] = float(np.mean(sorted_r[:idx_low]))

        # ─── ĐUÔI TRÊN (rủi ro chi phí cho buyer) ───
        idx_high = min(n - 1, int(np.ceil(n * cl)))
        out[f"Upper VaR {pct}%"]  = float(sorted_r[idx_high])
        out[f"Upper CVaR {pct}%"] = float(np.mean(sorted_r[idx_high:]))

    return out

# =================================================================
# 9. XÁC SUẤT VƯỢT NGƯỠNG CHO BUYER
# =================================================================
def buyer_threshold_probs(paths, s0, high_thr, low_thr):
    """
    Tính xác suất giá tháng 12 vượt ngưỡng:
      - P(giá > high_thr): rủi ro chi phí
      - P(giá < low_thr) : cơ hội mua rẻ
    """
    final_prices = paths[-1]
    p_high = float(np.mean(final_prices > high_thr))
    p_low  = float(np.mean(final_prices < low_thr))
    return p_high, p_low

# =================================================================
# 10. MAIN
# =================================================================
def main():
    rng = np.random.default_rng(SEED)

    # ── 10.1 Đọc & kiểm tra dữ liệu ─────────────────────────────
    hist_rets, hist_prices = load_data(
        INPUT_PATH, INPUT_SHEET, RETURN_COLUMN, PRICE_COLUMN
    )
    check_mean_reversion(hist_rets)

    s0 = float(hist_prices[-1])

    mu_m    = float(np.mean(hist_rets))
    sigma_m = float(np.std(hist_rets, ddof=1))
    mu_annual_base    = mu_m    * 12
    sigma_annual_base = sigma_m * np.sqrt(12)

    # Ngưỡng cho buyer
    risk_high_thr       = s0 * RISK_HIGH_MULT
    opportunity_low_thr = s0 * OPPORTUNITY_LOW_MULT

    print(f"\n{'='*65}")
    print(f"[INFO] Số điểm lịch sử    : {len(hist_rets)}")
    print(f"[INFO] S0 (giá bắt đầu)   : {s0:.4f} USD/lb")
    print(f"[INFO] mu  tháng / năm     : {mu_m:.6f} / {mu_annual_base:.6f}")
    print(f"[INFO] sig tháng / năm     : {sigma_m:.6f} / {sigma_annual_base:.6f}")
    print(f"[INFO] N_PERIODS={N_PERIODS} tháng, DT={DT:.5f} năm")
    print(f"[INFO] Ngưỡng RỦI RO chi phí   : {risk_high_thr:.3f} USD/lb (+{(RISK_HIGH_MULT-1)*100:.0f}%)")
    print(f"[INFO] Ngưỡng CƠ HỘI mua rẻ     : {opportunity_low_thr:.3f} USD/lb ({(OPPORTUNITY_LOW_MULT-1)*100:.0f}%)")
    print(f"{'='*65}\n")

    # ── 10.2 Lưới kịch bản ───────────────────────────────────────
    stat_rows        = []
    mean_paths_store = {}
    fan_data         = None

    sc = 0
    for mi, mu_a in enumerate(MU_ANNUAL_GRID):
        for si, sf in enumerate(SIG_FACTORS):
            sc += 1
            sigma_a = sigma_annual_base * sf

            paths, rets = run_gbm(
                N_SIMS, N_PERIODS, s0, mu_a, sigma_a, DT, rng
            )

            mean_path   = paths.mean(axis=1)
            scen_label  = f"S{sc:02d} (mu_a={mu_a:+.3f}, sig_a={sigma_a:.4f})"
            mean_paths_store[scen_label] = mean_path

            st = compute_stats(rets)
            p_high, p_low = buyer_threshold_probs(paths, s0, risk_high_thr, opportunity_low_thr)
            st.update({
                "Scenario": scen_label,
                "Mu_annual": mu_a,
                "Sigma_annual": sigma_a,
                "P(price>risk)": p_high,
                "P(price<opp)":  p_low,
            })
            stat_rows.append(st)

            # Fan chart dùng kịch bản trung tâm
            if mi == CENTER_MU_IDX and si == CENTER_SIG_IDX:
                fan_data = {
                    "period": np.arange(N_PERIODS + 1),
                    "P5" : np.percentile(paths,  5, axis=1),
                    "P25": np.percentile(paths, 25, axis=1),
                    "P50": np.percentile(paths, 50, axis=1),
                    "P75": np.percentile(paths, 75, axis=1),
                    "P95": np.percentile(paths, 95, axis=1),
                }

            print(f"[OK] {scen_label}  |  E[R]={st['E[Return]']:+.4f}  "
                  f"UpperVaR95={st['Upper VaR 95%']:+.4f}  "
                  f"P(risk)={p_high:.1%}  P(opp)={p_low:.1%}")

    # ── 10.3 Xuất Excel ──────────────────────────────────────────
    stat_df = pd.DataFrame(stat_rows)[[
        "Scenario", "Mu_annual", "Sigma_annual",
        "E[Return]", "Std[Return]", "Median",
        "Lower VaR 95%", "Lower VaR 99%", "Lower CVaR 95%", "Lower CVaR 99%",
        "Upper VaR 95%", "Upper VaR 99%", "Upper CVaR 95%", "Upper CVaR 99%",
        "P(price>risk)", "P(price<opp)",
        "Min", "Max",
    ]]
    mean_df = pd.DataFrame({"Period": np.arange(N_PERIODS + 1),
                              **mean_paths_store})
    fan_df  = pd.DataFrame(fan_data)

    with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
        stat_df.to_excel(writer, sheet_name="Statistics", index=False)
        mean_df.to_excel(writer, sheet_name="MeanPaths",  index=False)
        fan_df.to_excel( writer, sheet_name="FanChart",   index=False)

    print(f"\n[INFO] Đã xuất: {OUTPUT_PATH}")

    # ── 10.4 Chart ───────────────────────────────────────────────
    months = np.arange(N_PERIODS + 1)

    # Chart 1: Mean paths tất cả scenario
    fig1, ax1 = plt.subplots(figsize=(13, 6))
    for label, mp in mean_paths_store.items():
        # Đổi màu theo mu: dương = đỏ (rủi ro buyer), âm = xanh (cơ hội)
        mu_val = float(label.split("mu_a=")[1].split(",")[0])
        color = "crimson" if mu_val > 0 else ("seagreen" if mu_val < 0 else "steelblue")
        ax1.plot(months, mp, linewidth=1.2, color=color, alpha=0.6)
    ax1.axhline(s0, color="black", linewidth=1.5, linestyle="--",
                label=f"S0 = {s0:.3f}")
    ax1.axhline(risk_high_thr, color="red", linewidth=1.2, linestyle=":",
                label=f"Ngưỡng rủi ro chi phí = {risk_high_thr:.3f}")
    ax1.axhline(opportunity_low_thr, color="green", linewidth=1.2, linestyle=":",
                label=f"Ngưỡng cơ hội mua = {opportunity_low_thr:.3f}")
    ax1.set_title("Mean path – đỏ: giá tăng (rủi ro buyer) | xanh: giá giảm (cơ hội)")
    ax1.set_xlabel("Tháng")
    ax1.set_ylabel("Giá WW320 (USD/lb)")
    ax1.set_xticks(months)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8, loc="upper left")

    # Chart 2: Fan chart kịch bản trung tâm (góc nhìn buyer)
    fig2, ax2 = plt.subplots(figsize=(13, 6))
    x = fan_df["period"]
    ax2.fill_between(x, fan_df["P5"],  fan_df["P95"],
                     alpha=0.15, color="steelblue", label="P5–P95 (90%)")
    ax2.fill_between(x, fan_df["P25"], fan_df["P75"],
                     alpha=0.30, color="steelblue", label="P25–P75 (50%)")
    ax2.plot(x, fan_df["P50"], color="steelblue",
             linewidth=2.5, label="Median (P50)")

    # Vùng rủi ro (giá cao) cho buyer
    ax2.axhspan(risk_high_thr, fan_df["P95"].max() + 0.2,
                color="red", alpha=0.08, label="Vùng RỦI RO chi phí")
    # Vùng cơ hội (giá thấp) cho buyer
    ax2.axhspan(fan_df["P5"].min() - 0.2, opportunity_low_thr,
                color="green", alpha=0.08, label="Vùng CƠ HỘI mua rẻ")

    ax2.axhline(s0, color="black", linewidth=1.5, linestyle="--",
                label=f"S0 = {s0:.3f}")
    ax2.axhline(risk_high_thr, color="red", linewidth=1.5, linestyle="-",
                label=f"Rủi ro chi phí = {risk_high_thr:.3f}")
    ax2.axhline(opportunity_low_thr, color="green", linewidth=1.5, linestyle="-",
                label=f"Cơ hội mua = {opportunity_low_thr:.3f}")

    # In giá trị tại tháng 12
    for col, pct in [("P5","5%"),("P50","50%"),("P95","95%")]:
        val = fan_df[col].iloc[-1]
        ax2.annotate(f"{pct}: {val:.3f}",
                     xy=(12, val), xytext=(10.5, val),
                     fontsize=9,
                     arrowprops=dict(arrowstyle="-", color="gray"))

    ax2.set_title(
        f"Fan chart – góc nhìn BUYER – kịch bản trung tâm\n"
        f"mu_annual=0.00, sigma_annual={sigma_annual_base:.4f}"
    )
    ax2.set_xlabel("Tháng")
    ax2.set_ylabel("Giá WW320 (USD/lb)")
    ax2.set_xticks(months)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc="upper left")

    out_dir    = os.path.dirname(OUTPUT_PATH) or "."
    chart1_pth = os.path.join(out_dir, "mc_mean_paths_buyer.png")
    chart2_pth = os.path.join(out_dir, "mc_fan_chart_buyer.png")
    pdf_pth    = os.path.join(out_dir, "mc_charts_buyer.pdf")
    fig1.tight_layout(); fig1.savefig(chart1_pth, dpi=150)
    fig2.tight_layout(); fig2.savefig(chart2_pth, dpi=150)
    with PdfPages(pdf_pth) as pdf:
        pdf.savefig(fig1); pdf.savefig(fig2)

    print(f"[INFO] Chart 1 : {chart1_pth}")
    print(f"[INFO] Chart 2 : {chart2_pth}")
    print(f"[INFO] PDF     : {pdf_pth}")

    # ── 10.5 Tóm tắt kịch bản trung tâm (BUYER VIEW) ─────────────
    fan_end = {k: fan_df[k].iloc[-1] for k in ["P5","P25","P50","P75","P95"]}
    print(f"\n{'='*65}")
    print(f"KỊCH BẢN TRUNG TÂM – GIÁ THÁNG 12 (USD/lb) – GÓC NHÌN BUYER")
    print(f"{'='*65}")
    for k, v in fan_end.items():
        chg = (v / s0 - 1) * 100
        tag = ""
        if v >= risk_high_thr:
            tag = "  ← RỦI RO chi phí"
        elif v <= opportunity_low_thr:
            tag = "  ← CƠ HỘI mua rẻ"
        print(f"  {k:4s}: {v:.3f}  ({chg:+.1f}% vs S0){tag}")

    # Tìm kịch bản trung tâm trong stat_rows
    center_sc = [s for s in stat_rows
                 if s["Mu_annual"] == MU_ANNUAL_GRID[CENTER_MU_IDX]
                 and abs(s["Sigma_annual"] - sigma_annual_base * SIG_FACTORS[CENTER_SIG_IDX]) < 1e-8]
    if center_sc:
        st_c = center_sc[0]
        print(f"\n  ─── RỦI RO (giá TĂNG – xấu cho buyer) ───")
        print(f"  Upper VaR 95%      : {st_c['Upper VaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Upper VaR 95%']):.3f} USD/lb)")
        print(f"  Upper VaR 99%      : {st_c['Upper VaR 99%']:+.4f}  "
              f"(~{s0*(1+st_c['Upper VaR 99%']):.3f} USD/lb)")
        print(f"  Upper CVaR 95%     : {st_c['Upper CVaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Upper CVaR 95%']):.3f} USD/lb)")
        print(f"  Max (kịch bản đắt nhất) : {st_c['Max']:+.4f}  "
              f"(~{s0*(1+st_c['Max']):.3f} USD/lb)")

        print(f"\n  ─── CƠ HỘI (giá GIẢM – tốt cho buyer) ───")
        print(f"  Lower VaR 95%      : {st_c['Lower VaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Lower VaR 95%']):.3f} USD/lb)")
        print(f"  Lower VaR 99%      : {st_c['Lower VaR 99%']:+.4f}  "
              f"(~{s0*(1+st_c['Lower VaR 99%']):.3f} USD/lb)")
        print(f"  Lower CVaR 95%     : {st_c['Lower CVaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Lower CVaR 95%']):.3f} USD/lb)")
        print(f"  Min (kịch bản rẻ nhất)  : {st_c['Min']:+.4f}  "
              f"(~{s0*(1+st_c['Min']):.3f} USD/lb)")

        print(f"\n  ─── XÁC SUẤT VƯỢT NGƯỠNG ───")
        print(f"  P(giá > {risk_high_thr:.3f} – rủi ro chi phí) : {st_c['P(price>risk)']:.1%}")
        print(f"  P(giá < {opportunity_low_thr:.3f} – cơ hội mua rẻ): {st_c['P(price<opp)']:.1%}")
    print(f"{'='*65}\n")

if __name__ == "__main__":
    main()

[INFO] AR(1) autocorr returns: -0.179
[WARN] |rho|=0.179 > 0.15 → chuỗi có xu hướng mean-reversion.
       GBM giả định bước độc lập. Kết quả vẫn dùng được nhưng
       mô hình Ornstein–Uhlenbeck sẽ phản ánh sát hơn đặc trưng giá điều.

[INFO] Số điểm lịch sử    : 92
[INFO] S0 (giá bắt đầu)   : 3.2300 USD/lb
[INFO] mu  tháng / năm     : -0.000726 / -0.008709
[INFO] sig tháng / năm     : 0.039466 / 0.136713
[INFO] N_PERIODS=12 tháng, DT=0.08333 năm
[INFO] Ngưỡng RỦI RO chi phí   : 3.811 USD/lb (+18%)
[INFO] Ngưỡng CƠ HỘI mua rẻ     : 2.972 USD/lb (-8%)

[OK] S01 (mu_a=-0.080, sig_a=0.1025)  |  E[R]=-0.0773  UpperVaR95=+0.0861  P(risk)=0.6%  P(opp)=50.8%
[OK] S02 (mu_a=-0.080, sig_a=0.1196)  |  E[R]=-0.0762  UpperVaR95=+0.1168  P(risk)=1.8%  P(opp)=50.9%
[OK] S03 (mu_a=-0.080, sig_a=0.1367)  |  E[R]=-0.0763  UpperVaR95=+0.1445  P(risk)=3.2%  P(opp)=51.2%
[OK] S04 (mu_a=-0.080, sig_a=0.1709)  |  E[R]=-0.0770  UpperVaR95=+0.2053  P(risk)=6.4%  P(opp)=52.6%
[OK] S05 (mu_a=-0.080, sig_a=0.20

In [5]:
# -*- coding: utf-8 -*-
"""
Monte Carlo giá điều WW320 – BẢN BUYER (2 ĐUÔI)
================================================
(Giữ nguyên logic, chỉ nâng cấp Chart 1 – Mean Path cho legend sạch)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

# =================================================================
# 1. ĐƯỜNG DẪN  –  BẠN TỰ CHỈNH
# =================================================================
INPUT_PATH  = r"C:\Users\Acer\Desktop\Dan on food\Code\monte carlo\input.xlsx"
OUTPUT_PATH = r"C:\Users\Acer\Desktop\Dan on food\Code\monte carlo\ouput_buyer.xlsx"

RETURN_COLUMN = "% change "
PRICE_COLUMN  = "Average value"
INPUT_SHEET   = None

# =================================================================
# 2. THAM SỐ MONTE CARLO
# =================================================================
N_SIMS    = 50000
N_PERIODS = 12
DT        = 1 / 12
SEED      = 42

# =================================================================
# 3. LƯỚI QUÉT  –  MỞ RỘNG SANG MIỀN DƯƠNG
# =================================================================
MU_ANNUAL_GRID = [-0.08, -0.04, 0.00, +0.04, +0.08]
SIG_FACTORS    = [0.75, 0.875, 1.0, 1.25, 1.5]

CENTER_MU_IDX  = 2   # mu_annual = 0.00
CENTER_SIG_IDX = 2   # sig_factor = 1.0

# =================================================================
# 4. NGƯỠNG NGHIỆP VỤ CHO BUYER
# =================================================================
RISK_HIGH_MULT       = 1.18
OPPORTUNITY_LOW_MULT = 0.92

# =================================================================
# 5. ĐỌC DỮ LIỆU
# =================================================================
def load_data(path, sheet, ret_col, price_col):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(path, sheet_name=sheet if sheet else 0)
    elif ext == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Định dạng không hỗ trợ: {ext}")

    if ret_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột returns: '{ret_col}'. Cột hiện có: {list(df.columns)}")
    if price_col not in df.columns:
        raise ValueError(f"Không tìm thấy cột giá: '{price_col}'. Cột hiện có: {list(df.columns)}")

    returns = df[ret_col].dropna().astype(float).values
    prices  = df[price_col].dropna().astype(float).values

    if len(returns) < 12:
        raise ValueError("Cần ít nhất 12 điểm dữ liệu lịch sử.")
    return returns, prices

# =================================================================
# 6. KIỂM TRA MEAN REVERSION (AR1)
# =================================================================
def check_mean_reversion(returns):
    if len(returns) < 4:
        return
    rho = np.corrcoef(returns[:-1], returns[1:])[0, 1]
    print(f"[INFO] AR(1) autocorr returns: {rho:.3f}")
    if abs(rho) > 0.15:
        print(
            f"[WARN] |rho|={abs(rho):.3f} > 0.15 → chuỗi có xu hướng mean-reversion.\n"
            f"       GBM giả định bước độc lập. Kết quả vẫn dùng được nhưng\n"
            f"       mô hình Ornstein–Uhlenbeck sẽ phản ánh sát hơn đặc trưng giá điều."
        )

# =================================================================
# 7. GBM MỘT KỊCH BẢN
# =================================================================
def run_gbm(n_sims, n_periods, s0, mu_annual, sigma_annual, dt, rng):
    drift = (mu_annual - 0.5 * sigma_annual ** 2) * dt
    diff  = sigma_annual * np.sqrt(dt)

    z        = rng.standard_normal((n_periods, n_sims))
    log_inc  = drift + diff * z
    log_path = np.vstack([np.zeros(n_sims), np.cumsum(log_inc, axis=0)])
    paths    = s0 * np.exp(log_path)
    term_ret = paths[-1] / s0 - 1.0
    return paths, term_ret

# =================================================================
# 8. THỐNG KÊ 2 ĐUÔI
# =================================================================
def compute_stats(rets, conf_levels=(0.95, 0.99)):
    sorted_r = np.sort(rets)
    n = len(sorted_r)
    out = {
        "E[Return]"  : float(np.mean(rets)),
        "Std[Return]": float(np.std(rets, ddof=1)),
        "Median"     : float(np.median(rets)),
        "Min"        : float(sorted_r[0]),
        "Max"        : float(sorted_r[-1]),
    }
    for cl in conf_levels:
        alpha = 1 - cl
        pct   = int(round(cl * 100))

        idx_low = max(1, int(np.floor(n * alpha)))
        out[f"Lower VaR {pct}%"]  = float(sorted_r[idx_low - 1])
        out[f"Lower CVaR {pct}%"] = float(np.mean(sorted_r[:idx_low]))

        idx_high = min(n - 1, int(np.ceil(n * cl)))
        out[f"Upper VaR {pct}%"]  = float(sorted_r[idx_high])
        out[f"Upper CVaR {pct}%"] = float(np.mean(sorted_r[idx_high:]))

    return out

# =================================================================
# 9. XÁC SUẤT VƯỢT NGƯỠNG CHO BUYER
# =================================================================
def buyer_threshold_probs(paths, s0, high_thr, low_thr):
    final_prices = paths[-1]
    p_high = float(np.mean(final_prices > high_thr))
    p_low  = float(np.mean(final_prices < low_thr))
    return p_high, p_low

# =================================================================
# 10. MAIN
# =================================================================
def main():
    rng = np.random.default_rng(SEED)

    # ── 10.1 Đọc & kiểm tra dữ liệu ─────────────────────────────
    hist_rets, hist_prices = load_data(
        INPUT_PATH, INPUT_SHEET, RETURN_COLUMN, PRICE_COLUMN
    )
    check_mean_reversion(hist_rets)

    s0 = float(hist_prices[-1])

    mu_m    = float(np.mean(hist_rets))
    sigma_m = float(np.std(hist_rets, ddof=1))
    mu_annual_base    = mu_m    * 12
    sigma_annual_base = sigma_m * np.sqrt(12)

    risk_high_thr       = s0 * RISK_HIGH_MULT
    opportunity_low_thr = s0 * OPPORTUNITY_LOW_MULT

    print(f"\n{'='*65}")
    print(f"[INFO] Số điểm lịch sử    : {len(hist_rets)}")
    print(f"[INFO] S0 (giá bắt đầu)   : {s0:.4f} USD/lb")
    print(f"[INFO] mu  tháng / năm     : {mu_m:.6f} / {mu_annual_base:.6f}")
    print(f"[INFO] sig tháng / năm     : {sigma_m:.6f} / {sigma_annual_base:.6f}")
    print(f"[INFO] N_PERIODS={N_PERIODS} tháng, DT={DT:.5f} năm")
    print(f"[INFO] Ngưỡng RỦI RO chi phí   : {risk_high_thr:.3f} USD/lb (+{(RISK_HIGH_MULT-1)*100:.0f}%)")
    print(f"[INFO] Ngưỡng CƠ HỘI mua rẻ     : {opportunity_low_thr:.3f} USD/lb ({(OPPORTUNITY_LOW_MULT-1)*100:.0f}%)")
    print(f"{'='*65}\n")

    # ── 10.2 Lưới kịch bản ───────────────────────────────────────
    stat_rows        = []
    mean_paths_store = {}
    meta_store       = {}          # NEW: lưu (mu_a, sigma_a) theo label để vẽ chart
    fan_data         = None

    sc = 0
    for mi, mu_a in enumerate(MU_ANNUAL_GRID):
        for si, sf in enumerate(SIG_FACTORS):
            sc += 1
            sigma_a = sigma_annual_base * sf

            paths, rets = run_gbm(
                N_SIMS, N_PERIODS, s0, mu_a, sigma_a, DT, rng
            )

            mean_path   = paths.mean(axis=1)
            scen_label  = f"S{sc:02d} (mu_a={mu_a:+.3f}, sig_a={sigma_a:.4f})"
            mean_paths_store[scen_label] = mean_path
            meta_store[scen_label] = {"mu": mu_a, "sigma": sigma_a}

            st = compute_stats(rets)
            p_high, p_low = buyer_threshold_probs(paths, s0, risk_high_thr, opportunity_low_thr)
            st.update({
                "Scenario": scen_label,
                "Mu_annual": mu_a,
                "Sigma_annual": sigma_a,
                "P(price>risk)": p_high,
                "P(price<opp)":  p_low,
            })
            stat_rows.append(st)

            if mi == CENTER_MU_IDX and si == CENTER_SIG_IDX:
                fan_data = {
                    "period": np.arange(N_PERIODS + 1),
                    "P5" : np.percentile(paths,  5, axis=1),
                    "P25": np.percentile(paths, 25, axis=1),
                    "P50": np.percentile(paths, 50, axis=1),
                    "P75": np.percentile(paths, 75, axis=1),
                    "P95": np.percentile(paths, 95, axis=1),
                }

            print(f"[OK] {scen_label}  |  E[R]={st['E[Return]']:+.4f}  "
                  f"UpperVaR95={st['Upper VaR 95%']:+.4f}  "
                  f"P(risk)={p_high:.1%}  P(opp)={p_low:.1%}")

    # ── 10.3 Xuất Excel ──────────────────────────────────────────
    stat_df = pd.DataFrame(stat_rows)[[
        "Scenario", "Mu_annual", "Sigma_annual",
        "E[Return]", "Std[Return]", "Median",
        "Lower VaR 95%", "Lower VaR 99%", "Lower CVaR 95%", "Lower CVaR 99%",
        "Upper VaR 95%", "Upper VaR 99%", "Upper CVaR 95%", "Upper CVaR 99%",
        "P(price>risk)", "P(price<opp)",
        "Min", "Max",
    ]]
    mean_df = pd.DataFrame({"Period": np.arange(N_PERIODS + 1),
                              **mean_paths_store})
    fan_df  = pd.DataFrame(fan_data)

    with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
        stat_df.to_excel(writer, sheet_name="Statistics", index=False)
        mean_df.to_excel(writer, sheet_name="MeanPaths",  index=False)
        fan_df.to_excel( writer, sheet_name="FanChart",   index=False)

    print(f"\n[INFO] Đã xuất: {OUTPUT_PATH}")

    # ── 10.4 Chart ───────────────────────────────────────────────
    months = np.arange(N_PERIODS + 1)

    # =============================================================
    # CHART 1 – MEAN PATH (ĐÃ SỬA: màu theo mu, alpha theo sigma,
    #           legend đặt NGOÀI vùng vẽ nên không che đường)
    # =============================================================
    fig1, ax1 = plt.subplots(figsize=(14, 6.5))

    # 5 màu cố định cho 5 mức mu
    MU_COLORS = {
        -0.08: "#1f77b4",   # xanh dương đậm
        -0.04: "#2ca02c",   # xanh lá
         0.00: "#7f7f7f",   # xám
        +0.04: "#ff7f0e",   # cam
        +0.08: "#d62728",   # đỏ
    }

    # alpha theo sigma factor: nhỏ → nhạt, lớn → đậm
    SIG_ALPHAS = {0.75: 0.35, 0.875: 0.50, 1.0: 0.70, 1.25: 0.85, 1.5: 1.00}
    SIG_WIDTHS = {0.75: 1.0,  0.875: 1.2,  1.0: 1.5,  1.25: 1.8,  1.5: 2.1}

    # Vẽ từng đường – KHÔNG gán label ở đây (legend sẽ tạo thủ công)
    for label, mp in mean_paths_store.items():
        meta   = meta_store[label]
        mu_v   = meta["mu"]
        sf_v   = round(meta["sigma"] / sigma_annual_base, 3)

        # tìm key gần nhất trong dict
        sf_key = min(SIG_ALPHAS.keys(), key=lambda k: abs(k - sf_v))

        ax1.plot(
            months, mp,
            color   = MU_COLORS[mu_v],
            alpha   = SIG_ALPHAS[sf_key],
            linewidth = SIG_WIDTHS[sf_key],
            zorder  = 2,
        )

    # ─── Ngưỡng tham chiếu ───
    ax1.axhline(s0, color="black", linewidth=1.6, linestyle="--",
                zorder=3, label=f"S0 = {s0:.3f}")
    ax1.axhline(risk_high_thr, color="red", linewidth=1.4, linestyle=":",
                zorder=3, label=f"Ngưỡng RỦI RO chi phí = {risk_high_thr:.3f}")
    ax1.axhline(opportunity_low_thr, color="green", linewidth=1.4, linestyle=":",
                zorder=3, label=f"Ngưỡng CƠ HỘI mua = {opportunity_low_thr:.3f}")

    # ─── Legend 1: theo mu (5 dòng) – đặt ngoài phải ───
    legend_mu = [
        Line2D([0], [0], color=MU_COLORS[m], lw=2.2,
               label=f"mu = {m:+.2f}")
        for m in MU_ANNUAL_GRID
    ]
    # ─── Legend 2: theo sigma (5 dòng) – đặt ngoài phải, dưới legend mu ───
    legend_sig = [
        Line2D([0], [0], color="black", lw=SIG_WIDTHS[s],
               alpha=SIG_ALPHAS[s],
               label=f"sigma × {s:g}")
        for s in SIG_FACTORS
    ]
    # ─── Legend 3: ngưỡng tham chiếu ───
    legend_ref = [
        Line2D([0], [0], color="black", lw=1.6, ls="--", label=f"S0 = {s0:.3f}"),
        Line2D([0], [0], color="red",   lw=1.4, ls=":",  label=f"Rủi ro = {risk_high_thr:.3f}"),
        Line2D([0], [0], color="green", lw=1.4, ls=":",  label=f"Cơ hội = {opportunity_low_thr:.3f}"),
    ]

    # Gộp 3 legend thành 3 khối trong 1 legend ngoài
    handles = legend_mu + [Line2D([0],[0], color="none")] \
              + legend_sig + [Line2D([0],[0], color="none")] \
              + legend_ref
    labels  = ([h.get_label() for h in legend_mu] + [""]
               + [h.get_label() for h in legend_sig] + [""]
               + [h.get_label() for h in legend_ref])

    leg = ax1.legend(
        handles, labels,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),   # ← đẩy ra NGOÀI bên phải
        fontsize=8,
        frameon=True,
        borderpad=0.7,
        labelspacing=0.6,
        handlelength=2.2,
    )
    leg.get_frame().set_alpha(0.9)

    ax1.set_title("Mean path theo 25 kịch bản (màu = mu, đậm nhạt = sigma)")
    ax1.set_xlabel("Tháng")
    ax1.set_ylabel("Giá WW320 (USD/lb)")
    ax1.set_xticks(months)
    ax1.grid(True, alpha=0.3)

    # Chừa chỗ bên phải cho legend
    fig1.subplots_adjust(right=0.78)
    fig1.savefig(
        os.path.join(os.path.dirname(OUTPUT_PATH) or ".",
                     "mc_mean_paths_buyer.png"),
        dpi=150, bbox_inches="tight"
    )

    # =============================================================
    # CHART 2 – FAN CHART (giữ nguyên)
    # =============================================================
    fig2, ax2 = plt.subplots(figsize=(13, 6))
    x = fan_df["period"]
    ax2.fill_between(x, fan_df["P5"],  fan_df["P95"],
                     alpha=0.15, color="steelblue", label="P5–P95 (90%)")
    ax2.fill_between(x, fan_df["P25"], fan_df["P75"],
                     alpha=0.30, color="steelblue", label="P25–P75 (50%)")
    ax2.plot(x, fan_df["P50"], color="steelblue",
             linewidth=2.5, label="Median (P50)")

    ax2.axhspan(risk_high_thr, fan_df["P95"].max() + 0.2,
                color="red", alpha=0.08, label="Vùng RỦI RO chi phí")
    ax2.axhspan(fan_df["P5"].min() - 0.2, opportunity_low_thr,
                color="green", alpha=0.08, label="Vùng CƠ HỘI mua rẻ")

    ax2.axhline(s0, color="black", linewidth=1.5, linestyle="--",
                label=f"S0 = {s0:.3f}")
    ax2.axhline(risk_high_thr, color="red", linewidth=1.5, linestyle="-",
                label=f"Rủi ro chi phí = {risk_high_thr:.3f}")
    ax2.axhline(opportunity_low_thr, color="green", linewidth=1.5, linestyle="-",
                label=f"Cơ hội mua = {opportunity_low_thr:.3f}")

    for col, pct in [("P5","5%"),("P50","50%"),("P95","95%")]:
        val = fan_df[col].iloc[-1]
        ax2.annotate(f"{pct}: {val:.3f}",
                     xy=(12, val), xytext=(10.5, val),
                     fontsize=9,
                     arrowprops=dict(arrowstyle="-", color="gray"))

    ax2.set_title(
        f"Fan chart – góc nhìn BUYER – kịch bản trung tâm\n"
        f"mu_annual=0.00, sigma_annual={sigma_annual_base:.4f}"
    )
    ax2.set_xlabel("Tháng")
    ax2.set_ylabel("Giá WW320 (USD/lb)")
    ax2.set_xticks(months)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc="upper left")

    out_dir    = os.path.dirname(OUTPUT_PATH) or "."
    chart2_pth = os.path.join(out_dir, "mc_fan_chart_buyer.png")
    pdf_pth    = os.path.join(out_dir, "mc_charts_buyer.pdf")
    fig2.tight_layout(); fig2.savefig(chart2_pth, dpi=150)
    with PdfPages(pdf_pth) as pdf:
        pdf.savefig(fig1); pdf.savefig(fig2)

    print(f"[INFO] Chart 1 : {os.path.join(out_dir, 'mc_mean_paths_buyer.png')}")
    print(f"[INFO] Chart 2 : {chart2_pth}")
    print(f"[INFO] PDF     : {pdf_pth}")

    # ── 10.5 Tóm tắt kịch bản trung tâm (BUYER VIEW) ─────────────
    fan_end = {k: fan_df[k].iloc[-1] for k in ["P5","P25","P50","P75","P95"]}
    print(f"\n{'='*65}")
    print(f"KỊCH BẢN TRUNG TÂM – GIÁ THÁNG 12 (USD/lb) – GÓC NHÌN BUYER")
    print(f"{'='*65}")
    for k, v in fan_end.items():
        chg = (v / s0 - 1) * 100
        tag = ""
        if v >= risk_high_thr:
            tag = "  ← RỦI RO chi phí"
        elif v <= opportunity_low_thr:
            tag = "  ← CƠ HỘI mua rẻ"
        print(f"  {k:4s}: {v:.3f}  ({chg:+.1f}% vs S0){tag}")

    center_sc = [s for s in stat_rows
                 if s["Mu_annual"] == MU_ANNUAL_GRID[CENTER_MU_IDX]
                 and abs(s["Sigma_annual"] - sigma_annual_base * SIG_FACTORS[CENTER_SIG_IDX]) < 1e-8]
    if center_sc:
        st_c = center_sc[0]
        print(f"\n  ─── RỦI RO (giá TĂNG – xấu cho buyer) ───")
        print(f"  Upper VaR 95%      : {st_c['Upper VaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Upper VaR 95%']):.3f} USD/lb)")
        print(f"  Upper VaR 99%      : {st_c['Upper VaR 99%']:+.4f}  "
              f"(~{s0*(1+st_c['Upper VaR 99%']):.3f} USD/lb)")
        print(f"  Upper CVaR 95%     : {st_c['Upper CVaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Upper CVaR 95%']):.3f} USD/lb)")
        print(f"  Max (kịch bản đắt nhất) : {st_c['Max']:+.4f}  "
              f"(~{s0*(1+st_c['Max']):.3f} USD/lb)")

        print(f"\n  ─── CƠ HỘI (giá GIẢM – tốt cho buyer) ───")
        print(f"  Lower VaR 95%      : {st_c['Lower VaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Lower VaR 95%']):.3f} USD/lb)")
        print(f"  Lower VaR 99%      : {st_c['Lower VaR 99%']:+.4f}  "
              f"(~{s0*(1+st_c['Lower VaR 99%']):.3f} USD/lb)")
        print(f"  Lower CVaR 95%     : {st_c['Lower CVaR 95%']:+.4f}  "
              f"(~{s0*(1+st_c['Lower CVaR 95%']):.3f} USD/lb)")
        print(f"  Min (kịch bản rẻ nhất)  : {st_c['Min']:+.4f}  "
              f"(~{s0*(1+st_c['Min']):.3f} USD/lb)")

        print(f"\n  ─── XÁC SUẤT VƯỢT NGƯỠNG ───")
        print(f"  P(giá > {risk_high_thr:.3f} – rủi ro chi phí) : {st_c['P(price>risk)']:.1%}")
        print(f"  P(giá < {opportunity_low_thr:.3f} – cơ hội mua rẻ): {st_c['P(price<opp)']:.1%}")
    print(f"{'='*65}\n")

if __name__ == "__main__":
    main()

[INFO] AR(1) autocorr returns: -0.179
[WARN] |rho|=0.179 > 0.15 → chuỗi có xu hướng mean-reversion.
       GBM giả định bước độc lập. Kết quả vẫn dùng được nhưng
       mô hình Ornstein–Uhlenbeck sẽ phản ánh sát hơn đặc trưng giá điều.

[INFO] Số điểm lịch sử    : 92
[INFO] S0 (giá bắt đầu)   : 3.2300 USD/lb
[INFO] mu  tháng / năm     : -0.000726 / -0.008709
[INFO] sig tháng / năm     : 0.039466 / 0.136713
[INFO] N_PERIODS=12 tháng, DT=0.08333 năm
[INFO] Ngưỡng RỦI RO chi phí   : 3.811 USD/lb (+18%)
[INFO] Ngưỡng CƠ HỘI mua rẻ     : 2.972 USD/lb (-8%)

[OK] S01 (mu_a=-0.080, sig_a=0.1025)  |  E[R]=-0.0773  UpperVaR95=+0.0861  P(risk)=0.6%  P(opp)=50.8%
[OK] S02 (mu_a=-0.080, sig_a=0.1196)  |  E[R]=-0.0762  UpperVaR95=+0.1168  P(risk)=1.8%  P(opp)=50.9%
[OK] S03 (mu_a=-0.080, sig_a=0.1367)  |  E[R]=-0.0763  UpperVaR95=+0.1445  P(risk)=3.2%  P(opp)=51.2%
[OK] S04 (mu_a=-0.080, sig_a=0.1709)  |  E[R]=-0.0770  UpperVaR95=+0.2053  P(risk)=6.4%  P(opp)=52.6%
[OK] S05 (mu_a=-0.080, sig_a=0.20